# Projeto 3: Área Comercial - Análises de Vendas por Período

Performance de vendas e análises temporais

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

## 3.1 Criação da Dimensão de Tempo - gold.dm_tempo

In [0]:
# Get date range from pedidos
df_tempo = spark.sql("""
    SELECT 
        MIN(CAST(pedido_compra_timestamp AS DATE)) AS data_inicio,
        MAX(CAST(pedido_compra_timestamp AS DATE)) AS data_fim
    FROM silver.ft_pedidos
""")

# Create date dimension using correct column name
df_dim_tempo = spark.sql("""
    WITH date_range AS (
        SELECT 
            MIN(CAST(pedido_compra_timestamp AS DATE)) AS data_inicio,
            MAX(CAST(pedido_compra_timestamp AS DATE)) AS data_fim
        FROM silver.ft_pedidos
    )
    SELECT 
        date_val AS sk_tempo,
        YEAR(date_val) AS ano,
        QUARTER(date_val) AS trimestre,
        MONTH(date_val) AS mes,
        WEEKOFYEAR(date_val) AS semana_do_ano,
        DAY(date_val) AS dia,
        DAYOFWEEK(date_val) AS dia_da_semana_num,
        CASE DAYOFWEEK(date_val)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Segunda-feira'
            WHEN 3 THEN 'Terça-feira'
            WHEN 4 THEN 'Quarta-feira'
            WHEN 5 THEN 'Quinta-feira'
            WHEN 6 THEN 'Sexta-feira'
            WHEN 7 THEN 'Sábado'
        END AS dia_da_semana_nome,
        CASE MONTH(date_val)
            WHEN 1 THEN 'Janeiro'
            WHEN 2 THEN 'Fevereiro'
            WHEN 3 THEN 'Março'
            WHEN 4 THEN 'Abril'
            WHEN 5 THEN 'Maio'
            WHEN 6 THEN 'Junho'
            WHEN 7 THEN 'Julho'
            WHEN 8 THEN 'Agosto'
            WHEN 9 THEN 'Setembro'
            WHEN 10 THEN 'Outubro'
            WHEN 11 THEN 'Novembro'
            WHEN 12 THEN 'Dezembro'
        END AS mes_nome,
        CASE WHEN DAYOFWEEK(date_val) IN (1, 7) THEN 'Sim' ELSE 'Não' END AS eh_fim_de_semana
    FROM (
        SELECT explode(sequence(
            (SELECT data_inicio FROM date_range),
            (SELECT data_fim FROM date_range),
            INTERVAL 1 DAY
        )) AS date_val
    )
""")

df_dim_tempo.write.format("delta").mode("overwrite").saveAsTable("gold.dm_tempo")

In [0]:
print(f"Registros criados na dimensão tempo: {spark.table('gold.dm_tempo').count()}")
spark.table("gold.dm_tempo").show(5)

## 3.2 Criação da Fato gold.ft_vendas_geral

In [0]:
# Carregar tabelas necessárias
df_pedidos = spark.table("silver.ft_pedidos")
df_itens = spark.table("silver.ft_itens_pedidos")
df_avaliacoes = spark.table("silver.ft_avaliacoes_pedidos")
df_cotacao = spark.table("silver.dm_cotacao_dolar")

# Criar fato de vendas geral
df_vendas = df_itens.alias("i") \
    .join(df_pedidos.alias("p"), col("i.id_pedido") == col("p.id_pedido")) \
    .join(df_cotacao.alias("c"), to_date(col("p.pedido_compra_timestamp")) == col("c.data")) \
    .join(df_avaliacoes.alias("a"), col("p.id_pedido") == col("a.id_pedido"), "left") \
    .select(
        col("i.id_pedido"),
        col("i.id_item").cast("string"),
        col("p.id_consumidor").alias("fk_cliente"),
        col("i.id_produto").alias("fk_produto"),
        col("i.id_vendedor").alias("fk_vendedor"),
        to_date(col("p.pedido_compra_timestamp")).alias("fk_tempo"),
        col("p.status").alias("status_pedido"),
        col("p.tempo_entrega_dias").cast("int"),
        col("p.entrega_no_prazo"),
        col("i.preco_BRL").cast("decimal(12,2)").alias("valor_produto_brl"),
        col("i.preco_frete").cast("decimal(12,2)").alias("valor_frete_brl"),
        (col("i.preco_BRL") + col("i.preco_frete")).cast("decimal(12,2)").alias("valor_total_item_brl"),
        (col("i.preco_BRL") / col("c.cotacao_dolar")).cast("decimal(12,2)").alias("valor_produto_usd"),
        (col("i.preco_frete") / col("c.cotacao_dolar")).cast("decimal(12,2)").alias("valor_frete_usd"),
        ((col("i.preco_BRL") + col("i.preco_frete")) / col("c.cotacao_dolar")).cast("decimal(12,2)").alias("valor_total_item_usd"),
        col("c.cotacao_dolar").cast("decimal(8,4)"),
        col("a.avaliacao").cast("decimal(3,2)").alias("avaliacao_pedido")
    )

df_vendas.write.format("delta").mode("overwrite").saveAsTable("gold.ft_vendas_geral")


In [0]:
print(f"Registros criados: {spark.table('gold.ft_vendas_geral').count()}")
spark.table("gold.ft_vendas_geral").show(5)

## 3.3 Criação da view gold.view_vendas_por_periodo

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_vendas_por_periodo AS
    SELECT 
        t.ano,
        t.trimestre,
        t.mes,
        t.mes_nome,
        t.dia,
        t.dia_da_semana_num,
        COUNT(DISTINCT v.id_pedido) AS total_pedidos,
        COUNT(v.id_item) AS total_itens,
        CAST(SUM(v.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
        CAST(SUM(v.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
        CAST(AVG(v.valor_total_item_brl) AS DECIMAL(12,2)) AS ticket_medio_brl,
        CAST(AVG(v.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media
    FROM gold.ft_vendas_geral v
    INNER JOIN gold.dm_tempo t ON v.fk_tempo = t.sk_tempo
    WHERE v.status_pedido IN ('entregue', 'enviado')
    GROUP BY t.ano, t.trimestre, t.mes, t.mes_nome, t.dia, t.dia_da_semana_num
    ORDER BY t.ano, t.trimestre, t.mes, t.dia
""")

In [0]:
spark.sql("SELECT * FROM gold.view_vendas_por_periodo LIMIT 10").show()

### 3.3.1 Consultas Analíticas

#### Query 1: Dia da semana com maior receita total em reais

In [0]:
spark.sql("""
    SELECT 
        dia_da_semana_num,
        SUM(receita_total_brl) AS receita_total
    FROM gold.view_vendas_por_periodo
    GROUP BY dia_da_semana_num
    ORDER BY receita_total DESC
    LIMIT 1
""").show()

#### Query 2: Mês com maior ticket médio no último ano disponível

In [0]:
spark.sql("""
    WITH ultimo_ano AS (
        SELECT 2017 AS ano_max
    ),
    dados_filtrados AS (
        SELECT 
            t.mes,
            t.mes_nome,
            v.valor_total_item_brl,
            v.id_pedido
        FROM gold.ft_vendas_geral v
        INNER JOIN gold.dm_tempo t ON v.fk_tempo = t.sk_tempo
        CROSS JOIN ultimo_ano u
        WHERE t.ano = u.ano_max
          AND v.status_pedido IN ('entregue', 'enviado')
    )
    SELECT 
        mes,
        mes_nome,
        CAST(SUM(valor_total_item_brl) / COUNT(DISTINCT id_pedido) AS DECIMAL(12,2)) AS ticket_medio_real
    FROM dados_filtrados
    GROUP BY mes, mes_nome
    ORDER BY ticket_medio_real DESC
    LIMIT 1
""").show()


## 3.4 Criação da gold.view_top_produto

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_top_produto AS
    SELECT 
        v.fk_produto AS id_produto,
        p.categoria_produto,
        COUNT(v.id_item) AS quantidade_vendida,
        COUNT(DISTINCT v.id_pedido) AS total_pedidos,
        CAST(SUM(v.valor_produto_brl) AS DECIMAL(12,2)) AS receita_brl,
        CAST(SUM(v.valor_produto_usd) AS DECIMAL(12,2)) AS receita_usd,
        CAST(AVG(v.valor_produto_brl) AS DECIMAL(12,2)) AS preco_medio_brl,
        CAST(AVG(v.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media,
        CAST(AVG(p.peso_produto_gramas) AS DECIMAL(8,2)) AS peso_medio_gramas
    FROM gold.ft_vendas_geral v
    INNER JOIN silver.ft_produtos p ON v.fk_produto = p.id_produto
    GROUP BY v.fk_produto, p.categoria_produto
    ORDER BY receita_brl DESC
""")

In [0]:
spark.sql("SELECT * FROM gold.view_top_produto LIMIT 10").show(truncate=False)

## 3.5 Criação da view_vendas_produtos_esteticos (Fashion)

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_vendas_produtos_esteticos AS
    WITH vendas_fashion AS (
        SELECT 
            v.id_pedido,
            v.id_item,
            v.fk_tempo,
            v.valor_total_item_brl,
            v.valor_total_item_usd,
            v.avaliacao_pedido,
            p.categoria_produto
        FROM gold.ft_vendas_geral v
        INNER JOIN silver.ft_produtos p ON v.fk_produto = p.id_produto
        WHERE LOWER(p.categoria_produto) LIKE 'fashion%'
            AND v.status_pedido = 'entregue'
    )
    SELECT 
        t.ano,
        t.mes,
        vf.categoria_produto,
        COUNT(DISTINCT vf.id_pedido) AS total_pedidos,
        COUNT(vf.id_item) AS total_itens_vendidos,
        CAST(SUM(vf.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
        CAST(SUM(vf.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
        CAST(AVG(vf.valor_total_item_brl) AS DECIMAL(12,2)) AS ticket_medio_brl,
        CAST(AVG(vf.valor_total_item_usd) AS DECIMAL(12,2)) AS ticket_medio_usd,
        CAST(AVG(vf.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media
    FROM vendas_fashion vf
    INNER JOIN gold.dm_tempo t ON vf.fk_tempo = t.sk_tempo
    GROUP BY t.ano, t.mes, vf.categoria_produto
    ORDER BY t.ano, t.mes, receita_total_brl DESC
""")

In [0]:
spark.sql("SELECT * FROM gold.view_vendas_produtos_esteticos LIMIT 10").show(truncate=False)

## Validação

In [0]:
print("Objetos criados neste notebook:")
spark.sql("SHOW TABLES IN gold").filter(
    "tableName LIKE '%tempo%' OR tableName LIKE '%vendas%' OR tableName LIKE '%produto%' OR tableName LIKE '%estetico%'"
).show(truncate=False)